# step2 — 코드 관측 (RQ2)

**무엇을 확인하나:** 문맥에 있는 규약 이름(camelCase)과 위반 이름(snake_case)을, 모델이 새 함수 이름을
쓰려는 순간 **얼마나 보는지(어텐션)** 와 **얼마나 큰 내용을 담아 나르는지**를 층별로 잰다.
두 그룹의 차이가 '얼마나 보나'에 있는지 '담긴 내용'에 있는지 첫 단서를 얻는다. (원인 확정은 step3.)

**고정 설정:** 4모델 · 함수이름 504개(42묶음) · 무작위값 42 · 문맥은 규약 6 + 위반 6 균형.

> **메모리 주의(T4):** 이 실험은 어텐션을 다 꺼내야 해서 메모리를 많이 쓴다.
> **deepseek-6.7b는 T4에서 메모리 초과(OOM)가 날 수 있다** → 그러면 셀 ④에서 그 모델에 8bit를 켠다(주석 참고).

**모델 하나씩 돌린다.** 셀 ④ 맨 위 `PICK`에서 모델 1개 고르고 → 셀 ⑤~⑦. 끝나면 `PICK` 바꿔 반복.
끊겨도 이미 저장된 건 건너뛴다.


In [ ]:
# 환경 설정 — 설치, GPU 확인, 무작위값 42 고정
!pip install -q transformers accelerate torch matplotlib pandas bitsandbytes

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('무작위값 고정:', SEED)


In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
BRANCH = 'integration/step1-5'
!git fetch --quiet origin $BRANCH
!git checkout $BRANCH
!git pull --quiet origin $BRANCH
!pip install -e . -q
import sys; sys.path.insert(0, 'src')


In [ ]:
# 조건 설정 — 4모델 중 하나 골라 42묶음 관측
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]

# ★ 이번 세션에 돌릴 모델 하나만 고른다 (0=qwen, 1=deepseek, 2=llama, 3=stable).
PICK = 0
MODEL = MODELS[PICK]

# deepseek-6.7b가 T4에서 메모리 초과나면 아래 한 줄 주석 해제(8bit):
# if MODEL.family == 'deepseek': MODEL = ModelSpec(name=MODEL.name, family='deepseek', dtype='float16', quantization='8bit')

BLOCKS = list(range(42))            # 504 이름 전부 커버

def observe_cond(block):
    return Condition(
        model=MODEL,
        preceding=PrecedingCode(n_compliant=6, n_functions=12, composition=Composition.POOL,
                                pool_block=block),   # 규약 6 + 위반 6 균형
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL),
        seed=42,
    )

conditions = [observe_cond(b) for b in BLOCKS]
print('모델:', MODEL.family, '| 조건 수(묶음):', len(conditions))
print('예:', conditions[0].slug())


In [ ]:
# 실행 — 관측. 조건마다 즉시 저장(재개). 어텐션을 꺼내야 해서 eager로 로드.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
import numpy as np

STEP = 'step2_code-observe'
todo = [c for c in conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 전체 {len(conditions)} / 남은 {len(todo)}')

if todo:
    handle = load_model(MODEL, attn_implementation='eager')   # 어텐션 관측엔 eager 필수
    print(f'  층수 {handle.num_layers} | GQA {handle.gqa_info()}')
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle, mode='observe')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ2'))
        if i % 7 == 0 or i == len(todo):
            # 중간 결과: 지금 조건에서 각 층 최고 어텐션이 규약/위반 중 어느 쪽인지 대략
            pl = out.metrics.per_layer
            camel = np.mean([v.get('code_camel__attention_weight', 0) for v in pl.values()])
            snake = np.mean([v.get('code_snake__attention_weight', 0) for v in pl.values()])
            print(f'    [{i}/{len(todo)}] 평균 어텐션  규약(camel) {camel:.4f}  vs  위반(snake) {snake:.4f}')
    print('  완료.')
else:
    print('  이미 다 됨 — 건너뜀')


In [ ]:
# 결과 로드 (이 모델 것)
from harness import result_path
from harness.results import load_result
recs = [load_result(result_path(c, step='step2_code-observe')) for c in conditions
        if result_path(c, step='step2_code-observe').exists()]
print('불러온 묶음:', len(recs), '-> results/step2_code-observe/')


In [ ]:
STEPS = ['step2_code-observe']
# ⑧ 결과 zip으로 묶어 내려받기
import shutil, os, glob

def pack(step):
    d = f'results/{step}'
    if not os.path.isdir(d):
        print(f'  [건너뜀] {d} 폴더가 없다 — 이 스텝은 아직 안 돌렸다')
        return None
    n = len(glob.glob(f'{d}/*.json'))
    if n == 0:
        print(f'  [건너뜀] {d} 가 비어 있다')
        return None
    path = shutil.make_archive(step, 'zip', d)
    print(f'  {step}: {n}개 → {path} ({os.path.getsize(path)/1e6:.1f}MB)')
    return path

print('results/ 안에 있는 폴더:', sorted(os.listdir('results')) if os.path.isdir('results') else '(results 폴더 없음)')
print()
made = [p for p in (pack(s) for s in STEPS) if p]

if not made:
    print('\n내려받을 것이 없다. 실행 셀을 먼저 돌렸는지 확인할 것.')
else:
    try:
        from google.colab import files
        for p in made:
            files.download(p)
        print('\n다운로드 시작. 브라우저가 막으면 왼쪽 **파일 탐색기**에서 직접 받으면 된다.')
    except Exception as e:
        print(f'\nColab 자동 다운로드 불가({type(e).__name__}). 위 경로에서 직접 받을 것.')
